# Overfitting and Regularization

In this experiment, the effect of dropout regularization on a feedforward neural network is investigated. Two identical neural networks are trained on the MNIST handwritten digit dataset. The first network serves as a baseline and contains no dropout layers, while the second includes dropout with a probability of 0.2 after each hidden layer.

Training and validation accuracy are recorded after every epoch for both models. The resulting learning curves are compared to determine whether dropout reduces overfitting by decreasing the gap between training and validation performance.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Fetch MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

## Training and validation split
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [54000, 6000],
    generator=generator
)

test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

## Define the Training Loop

In [ ]:
def train_model(model, optimizer, criterion, train_loader, val_loader, epochs):

    train_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0 

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()

            loss.backward()
            optimizer.step()

        average_loss = running_loss / len(train_loader)
        train_losses.append(average_loss)

        ## Evaluate the model on the training set to compute accuracy
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in train_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_accuracies.append(train_accuracy)

        ## Evaluate the model on the validation set to compute accuracy
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Loss: {average_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )
    return train_losses, train_accuracies, val_accuracies

## Create the Baseline and Dropout Models

In [ ]:
## Baseline model
class FeedForwardNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        return x

## Dropout model
class FeedForwardDropoutNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 128)
        ## Incorporates 20% dropout on first 2 layers so that 20% of the neurons are randomly turned off during training to prevent overfitting
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)

        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = torch.relu(x)
        x = self.dropout2(x)

        x = self.fc3(x)

        return x        

## Train the Models

In [ ]:
num_epochs = 10

baseline_model = FeedForwardNN().to(device)

criterion = nn.CrossEntropyLoss()

baseline_optimizer = optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)

baseline_loss, baseline_train_acc, baseline_val_acc = train_model(
    baseline_model,
    baseline_optimizer,
    criterion,
    train_loader,
    val_loader,
    num_epochs
)

dropout_model = FeedForwardDropoutNN().to(device)

dropout_optimizer = optim.Adam(
    dropout_model.parameters(),
    lr=0.001
)

dropout_loss, dropout_train_acc, dropout_val_acc = train_model(
    dropout_model,
    dropout_optimizer,
    criterion,
    train_loader,
    val_loader,
    num_epochs
)


## Visualizing Accuracy and Loss

In [ ]:
## Loss graph
plt.figure(figsize=(8,5))

plt.plot(baseline_loss, label="Baseline")
plt.plot(dropout_loss, label="Dropout")

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)

plt.show()

## Accuracy graph
plt.figure(figsize=(8,5))

plt.plot(baseline_train_acc, label="Baseline Train")
plt.plot(baseline_val_acc, label="Baseline Validation")

plt.plot(dropout_train_acc, label="Dropout Train")
plt.plot(dropout_val_acc, label="Dropout Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

print("Final Results")
print(f"Baseline Train Accuracy:    {baseline_train_acc[-1]:.2f}%")
print(f"Baseline Validation Accuracy: {baseline_val_acc[-1]:.2f}%")
print()

print(f"Dropout Train Accuracy:     {dropout_train_acc[-1]:.2f}%")
print(f"Dropout Validation Accuracy:  {dropout_val_acc[-1]:.2f}%")

## Evaluate Models on the Test Set

In [ ]:
def evaluate_model(model, test_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy

baseline_test_accuracy = evaluate_model(
    baseline_model,
    test_loader
)

dropout_test_accuracy = evaluate_model(
    dropout_model,
    test_loader
)

print("Final Results")
print("-------------------------")
print(f"Baseline Test Accuracy: {baseline_test_accuracy:.2f}%")
print(f"Dropout Test Accuracy:  {dropout_test_accuracy:.2f}%")

## Final Results

Overall, both models achieved over 97% accuracy on the test set, indicating that each was able to classify handwritten digits effectively. The baseline model achieved a lower training loss, reflecting its ability to fit the training data more closely. However, the dropout model achieved slightly higher validation accuracy throughout training and also produced a slightly higher final test accuracy. This suggests that the baseline model exhibited greater signs of overfitting, while the dropout model generalized better to previously unseen data.

Since the dropout model randomly disabled 20% of the neurons during each pass, it prevented the model from being overly dependent on a few neurons. Because of this, the dropout model was encouraged to learn more robust and generalizable features rather than memorizing patterns in the data. As a result, the dropout model was able to use these generalizations to perform slightly better on unseen data.